## Before running

A virtual environment can be created using 
- 'pipenv install'
- 'pipenv shell'

This will allow us to all use the same packages and versions. They are listed in the Pipfile

In [3]:
from GeneticAlgorithm import *

2022-11-22 10:04:06.822414: I tensorflow/core/platform/cpu_feature_guard.cc:193] This TensorFlow binary is optimized with oneAPI Deep Neural Network Library (oneDNN) to use the following CPU instructions in performance-critical operations:  AVX2 FMA
To enable them in other operations, rebuild TensorFlow with the appropriate compiler flags.


## Inputs

Dictionaries are taken as input from a parameter file, they contain the parameters for each soap descriptor

In [ ]:
descDict1 = {'lower': 1, 'upper': 50, 'centres': '{8, 7, 6, 1, 16, 17, 9}',
             'neighbours': '{8, 7, 6, 1, 16, 17, 9}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50, 
             'min_cutoff': 1, 'max_cutoff': 50, 'min_sigma': 0.1, 
             'max_sigma': 0.9, 'message_steps': 0}

descDict2 = {'lower': 51, 'upper': 100, 'centres': '{8, 7, 6, 1, 16, 17, 9}',
             'neighbours': '{8, 7, 6, 1, 16, 17, 9}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50,
             'min_cutoff': 51, 'max_cutoff': 100, 'min_sigma': 1.1, 
             'max_sigma': 1.9, 'message_steps': 0}

Other parameters are also taken as input. These are automatically checked that the parameters are viable

In [ ]:
num_gens = 100
best_sample, lucky_few, population_size, number_of_children = 4, 2, 12, 4
early_stop = 2
early_number = 3 
min_generations = 5

## GeneParameter

GeneParameter class is created from each descriptor dictionary. 

In [ ]:
params1 = GeneParameters(**descDict1)
params2 = GeneParameters(**descDict2)

In [ ]:
params1

## GeneSet

We can use these classes to create a specific set of parameters that are consistant with these values. This returns a randomly generated GeneSet class

In [ ]:
example_gene_set = params1.make_gene_set()
example_gene_set

We can get the parameters used to create the GeneSet class

In [ ]:
example_gene_set.gene_parameters

We can get a descriptor string to be used as an input for getting SOAPs

In [ ]:
example_gene_set.get_soap_string()

We can also mutate the gene using the mutation chance in the GeneParameters class

In [ ]:
print(f"Before mutation {example_gene_set}")
example_gene_set.mutate_gene()
print(f"After mutation {example_gene_set}")

## Individual

An Individual is made up of a list of GeneSet classes.

In [ ]:
example_gene_set_two = params2.make_gene_set()
gene_set_list = [example_gene_set, example_gene_set_two]
example_individual = Individual(gene_set_list)
example_individual

Getting the score for an indivudual

In [ ]:
gene_set_list[0].cutoff = 7
gene_set_list[0].l_max = 5
gene_set_list[0].n_max = 8
example_individual = Individual(gene_set_list[:1])

In [ ]:
example_individual.get_score()
example_individual.score

Breeding two individuals to create a child. Mutation is automatically performed during this

In [ ]:
example_individual_two = Individual(gene_set_list)
print(f"Breeding {example_individual} with {example_individual_two}")
child = breed_individuals(example_individual, example_individual_two)
print(f"Created child {child}")

## Population

A Population is a collection of Individual classes. This can be created using a list of GeneParameter classes

In [ ]:
gene_parameters = [params1, params2]
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)
pop

To initialise the population

In [ ]:
pop.initialise_population()

If you want a way of neatly seeing what individuals are in the population

In [ ]:
pop.print_population()

The next generation can then be generated 

In [ ]:
pop.next_generation()
pop.print_population()

So to run the full GA 

In [ ]:
for _ in range(num_gens):
    pop.next_generation()
pop.print_population()

## BestHistory

BestHistory is a class to store the history and check convergence criteria. So the entire GA can be run, printed, and saved using the following code snippet:

In [ ]:
hist = BestHistory(early_stop, early_number, min_generations)
pop = Population(best_sample, lucky_few, population_size, 
                 number_of_children, gene_parameters, 
                 maximise_scores = True)

pop.initialise_population()    
for gen in range(num_gens):
    if hist.converged:
        break
    print(f"Generation {gen}")
    pop.next_generation()
    hist.append(pop)
    print("-------")

There now exists the entire history of the best Individuals throughout each generation that can be saved and easily accessed. 

In [ ]:
vars(hist)

### TESTING MP

In [34]:
from GeneticAlgorithm import *
descDict1 = {'lower': 1, 'upper': 50, 'centres': '{8, 7, 6, 1, 16, 17, 9}',
             'neighbours': '{8, 7, 6, 1, 16, 17, 9}', 'mu': 0, 
             'mu_hat': 0, 'nu': 2, 'nu_hat': 0, 'mutation_chance': 0.50, 
             'min_cutoff': 1, 'max_cutoff': 50, 'min_sigma': 0.1, 
             'max_sigma': 0.9, 'message_steps':0}
params1 = GeneParameters(**descDict1)
example_gene_set = params1.make_gene_set()

conf_s, data = get_conf()

example_gene_set.cutoff = 5
example_gene_set.l_max = 3
example_gene_set.n_max = 6
test_ind = Individual([example_gene_set])

conf_s file exists


In [35]:
test_ind.comp_soaps(conf_s, data)

Getting soap for Atoms(symbols='COC6NCOC4ClC7O2C2O2H18', pbc=False)
Getting soap for Atoms(symbols='C2ONC6OH9', pbc=False)
Getting soap for Atoms(symbols='C3SO2C6FCONC7NCF3OH14', pbc=False)
Getting soap for Atoms(symbols='C6NC6NC3OC7NOH25', pbc=False)
Getting soap for Atoms(symbols='C25H34O6', pbc=False)
Getting soap for Atoms(symbols='C3SCONC5O2H15', pbc=False)
Getting soap for Atoms(symbols='COC6OC2NC3OC12NOH26', pbc=False)
Getting soap for Atoms(symbols='C10N2C6SO2NCF3H14', pbc=False)
Getting soap for Atoms(symbols='C9ONCOCCl2ONO2H12', pbc=False)
Getting soap for Atoms(symbols='C2NC2NC22H28', pbc=False)
Getting soap for Atoms(symbols='C14ClOC6NCH26', pbc=False)
Getting soap for Atoms(symbols='C19ClNC2NCH17', pbc=False)
Getting soap for Atoms(symbols='C12OC10NOH27', pbc=False)
Getting soap for Atoms(symbols='C2OCSCONC3NCONFH10', pbc=False)
Getting soap for Atoms(symbols='C10OC8OH24', pbc=False)
Getting soap for Atoms(symbols='C9ONC6FC9FO2H21', pbc=False)
Getting soap for Atoms(symbol

In [28]:
test_ind.get_score()

conf_s file exists
Getting soap for Atoms(symbols='COC6NCOC4ClC7O2C2O2H18', pbc=False)
Getting soap for Atoms(symbols='C2ONC6OH9', pbc=False)
Getting soap for Atoms(symbols='C3SO2C6FCONC7NCF3OH14', pbc=False)
Getting soap for Atoms(symbols='C6NC6NC3OC7NOH25', pbc=False)
Getting soap for Atoms(symbols='C25H34O6', pbc=False)
Getting soap for Atoms(symbols='C3SCONC5O2H15', pbc=False)
Getting soap for Atoms(symbols='COC6OC2NC3OC12NOH26', pbc=False)
Getting soap for Atoms(symbols='C10N2C6SO2NCF3H14', pbc=False)
Getting soap for Atoms(symbols='C9ONCOCCl2ONO2H12', pbc=False)
Getting soap for Atoms(symbols='C2NC2NC22H28', pbc=False)
Getting soap for Atoms(symbols='C14ClOC6NCH26', pbc=False)
Getting soap for Atoms(symbols='C19ClNC2NCH17', pbc=False)
Getting soap for Atoms(symbols='C12OC10NOH27', pbc=False)
Getting soap for Atoms(symbols='C2OCSCONC3NCONFH10', pbc=False)
Getting soap for Atoms(symbols='C10OC8OH24', pbc=False)
Getting soap for Atoms(symbols='C9ONC6FC9FO2H21', pbc=False)
Getting so

/Users/trentbarnard/.local/share/virtualenvs/SOAP_GAS_TMS-fhxFTjkw/lib/python3.9/site-packages/scipy/stats/_stats_py.py:4427: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))
/Users/trentbarnard/.local/share/virtualenvs/SOAP_GAS_TMS-fhxFTjkw/lib/python3.9/site-packages/scipy/stats/_stats_py.py:4427: ConstantInputWarning: An input array is constant; the correlation coefficient is not defined.
  warnings.warn(stats.ConstantInputWarning(msg))


In [29]:
y_train = test_ind.results_dictionary['y_train_actual'][1]
y_train_pred = test_ind.results_dictionary['y_train_predictions'][1]
y_test = test_ind.results_dictionary['y_test_actual'][1]
y_test_pred = test_ind.results_dictionary['y_test_predictions'][1]

In [30]:
testMSE = mean_squared_error(y_test, y_test_pred)
trainMSE =  mean_squared_error(y_train, y_train_pred)
testCorr = pearsonr(y_test, y_test_pred)[0]
trainCorr = pearsonr(y_train, y_train_pred)[0]

In [31]:
testMSE

7503037176.216454

In [32]:
y_test

array([323., 331., 338., 373., 270., 382., 335., 309., 348., 263., 416.,
       322., 334., 264., 245., 293., 324., 366., 274., 334., 277., 279.,
       284., 286., 308.])

In [33]:
y_test_pred

array([5.6208328e+04, 7.0786961e+04, 2.8960522e+02, 4.2389656e+05,
       2.8578485e+02, 3.3207755e+02, 3.0276480e+02, 3.0367703e+02,
       1.1140639e+04, 2.9385080e+02, 3.2841888e+02, 2.9679028e+02,
       2.9601764e+02, 2.7312000e+02, 3.0969043e+02, 2.7836551e+02,
       3.0875635e+02, 3.1404138e+02, 3.0193774e+02, 2.9627243e+02,
       2.9682938e+02, 3.0251486e+02, 2.8653192e+02, 3.0098972e+02,
       2.8837280e+02], dtype=float32)

In [36]:
test_ind.results_dictionary

defaultdict(list, {})